In [ ]:
class RegNetWithFCA(nn.Module):
    def __init__(self, num_classes, model_name='regnetx_040'):
        super(RegNetWithFCA, self).__init__()
        self.feature_extractor = create_model(model_name, pretrained=False, features_only=True)
        self.attention_blocks = nn.ModuleList([
            FrequencyChannelAttention(channels) for channels in self.feature_extractor.feature_info.channels()
        ])

        # Classifier
        in_features = self.feature_extractor.feature_info.channels()[-1]
        self.classifier = nn.Linear(in_features, num_classes)

    def forward(self, x):
        # Extract feature maps
        features = self.feature_extractor(x)  # Returns a list of feature maps

        # Apply ECA attention to each feature map
        for i, (feature_map, attention_block) in enumerate(zip(features, self.attention_blocks)):
            features[i] = attention_block(feature_map)  # Apply ECAAttention

        # Use the last feature map (after attention) for classification
        x = features[-1].mean(dim=[2, 3])  # Global average pooling
        x = self.classifier(x)  # Classifier layer
        return x